In [1]:
import jax.numpy as jnp 
import jax as j
import matplotlib.pyplot as plt


In [2]:
A = 1.0 
alpha = 0.7 
gamma = 0.8
epsilon = 0.08
v0 = 0.0
w0 = 0.0
d = 0.01 #difusao


h = 0.1 #passos de tempo
delta_x = 0.1 #passo espacial

xf = 1.0
tf = 20.0 #tempo final

t = jnp.arange(0, tf + h, h)
x = jnp.arange(0, xf + delta_x, delta_x)



In [ ]:
def I_ion(v,w,t,iamp):
    iapp = jnp.where((t <= 0.1) & (v < 0.0), iamp, 0.0)
    return A*v*(v-alpha)*(1-v) - w + iapp

def g(v,w):
    return epsilon*(v + alpha - gamma*w) 

estimulo = jnp.zeros(len(x))

for i in range(len(x)):
    if x[i] <= 0.1:
        estimulo.at[i].set(5.0)
    else:
        estimulo.at[i].set(0.0)



def passo(estado,t,estimulo):
    V, W = estado
    dvdt = d*(V[2:] - 2*V[1:-1] +V[:-2])/delta_x**2 + I_ion(V[1:-1], W[1:-1], t, estimulo[1:-1]) 
    dwdt = g(V[1:-1], W[1:-1])

    V = V.at[1:-1].set(V[1:-1] + dvdt * h)
    W = W.at[1:-1].set(W[1:-1] + dwdt * h)

    V = V.at[0].set(V[1])
    V = V.at[-1].set(V[-2])
   
    W = W.at[0].set(W[0]+h*g(V[0], W[0]))
    W = W.at[-1].set(W[-1]+h*g(V[-1], W[-1]))

    return (V, W) , (V, W)


solucao = jnp.zeros((len(t), len(x), 2)) # 2 para V e W
solucao = j.lax.scan(passo,(v0,w0), t, estimulo)



V = jnp.zeros((len(t), len(x)))
W = jnp.zeros((len(t), len(x)))
V = V.at[0,:].set(-1.2)
W = W.at[0,:].set(-0.62)

TypeError: Only scalar arrays can be converted to Python scalars; got arr.ndim=1